# 08 — Pricing Evaluation

Evaluate all four models on real held-out Kaggle options data (2020-2023):
- Transformer A (Heston only)
- Transformer B (GAN only)
- Transformer C (Mixed)
- Black-Scholes baseline

Report MAE, MSE, MAPE broken down by:
- Volatility regime (low / medium / high)
- Moneyness (OTM / ATM / ITM)
- Time to expiry (short / medium / long)

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import torch
sys.path.insert(0, os.path.abspath('..'))

from src.models.transformer import build_transformer
from src.evaluation.pricing_eval import evaluate_all_models

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
CHECKPOINT_BASE = os.path.join('..', 'checkpoints', 'transformer')
FIGURES_DIR = os.path.join('..', 'results', 'figures')
TABLES_DIR = os.path.join('..', 'results', 'tables')

device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

## Load Test Data and Models

In [ ]:
# Load test options
opts_test = pd.read_pickle(os.path.join(PROCESSED_DIR, 'opts_test.pkl'))
print(f'Test options: {len(opts_test)} contracts')
print(f'Date range: {opts_test["date"].min()} to {opts_test["date"].max()}')

# Load master DF for BS rv_21d lookup
master_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'master_df.csv'), index_col=0, parse_dates=True)

In [ ]:
# Load trained models
models = {}

for exp_name, exp_dir in [('Transformer A (Heston)', 'experiment_a'),
                           ('Transformer B (GAN)', 'experiment_b'),
                           ('Transformer C (Mixed)', 'experiment_c')]:
    ckpt_path = os.path.join(CHECKPOINT_BASE, exp_dir, 'best_model.pt')
    if os.path.exists(ckpt_path):
        model = build_transformer()
        model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
        model.eval()
        models[exp_name] = model
        print(f'Loaded {exp_name}')
    else:
        print(f'{exp_name}: checkpoint not found at {ckpt_path}')

# Black-Scholes baseline
models['Black-Scholes'] = 'BS'
print(f'\nTotal models to evaluate: {len(models)}')

## Run Full Evaluation

In [ ]:
results = evaluate_all_models(
    opts_test, models,
    master_df=master_df,
    device=device,
    figures_dir=FIGURES_DIR,
    tables_dir=TABLES_DIR,
)

## Detailed Analysis: Where Models Differ Most

Black-Scholes is known to fail most severely for **short-dated ATM options in high-vol regimes**.

In [ ]:
import matplotlib.pyplot as plt

# Load detailed results
regime_path = os.path.join(TABLES_DIR, 'regime_metrics.csv')
money_path = os.path.join(TABLES_DIR, 'moneyness_metrics.csv')
expiry_path = os.path.join(TABLES_DIR, 'expiry_metrics.csv')

if os.path.exists(regime_path):
    regime_df = pd.read_csv(regime_path)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for i, regime in enumerate(['Low (VIX<15)', 'Medium (15-25)', 'High (VIX>25)']):
        subset = regime_df[regime_df['Regime'] == regime]
        if len(subset):
            axes[i].bar(subset['Model'], subset['MAE'])
            axes[i].set_title(f'MAE — {regime}')
            axes[i].tick_params(axis='x', rotation=30)
    
    plt.suptitle('MAE by Volatility Regime')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'eval_regime_breakdown.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if os.path.exists(money_path):
    money_df = pd.read_csv(money_path)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for i, bucket in enumerate(['OTM', 'ATM', 'ITM']):
        subset = money_df[money_df['Moneyness'] == bucket]
        if len(subset):
            axes[i].bar(subset['Model'], subset['MAE'])
            axes[i].set_title(f'MAE — {bucket}')
            axes[i].tick_params(axis='x', rotation=30)
    
    plt.suptitle('MAE by Moneyness')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'eval_moneyness_breakdown.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if os.path.exists(expiry_path):
    expiry_df = pd.read_csv(expiry_path)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for i, bucket in enumerate(['short', 'medium', 'long']):
        subset = expiry_df[expiry_df['Expiry'] == bucket]
        if len(subset):
            axes[i].bar(subset['Model'], subset['MAE'])
            axes[i].set_title(f'MAE — {bucket} expiry')
            axes[i].tick_params(axis='x', rotation=30)
    
    plt.suptitle('MAE by Time to Expiry')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'eval_expiry_breakdown.png'), dpi=150, bbox_inches='tight')
    plt.show()

## Summary Table

In [ ]:
print('\n' + '=' * 60)
print('FINAL RESULTS: Overall Test Set Performance (2020-2023)')
print('=' * 60)
print(results.round(4).to_string())